In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
dbutils.widgets.text("catalog", "workspace")

catalog = dbutils.widgets.get("catalog").strip()

In [0]:
import os, sys, yaml
from pyspark.sql import functions as F

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from nyctaxi.gold import *

with open("../config/dimensions.yml") as f:
    dims = yaml.safe_load(f)

GOLD = f"{catalog}.gold"
trips = spark.table(f"{catalog}.silver.trips")
zones = spark.table(f"{catalog}.silver.taxi_zones")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def write(df, name):
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD}.{name}"))
    print(f"{GOLD}.{name}: {spark.table(f'{GOLD}.{name}').count():,} rows")

In [0]:
write(build_code_dim(spark, dims["payment"],  "payment_id",  "payment_desc"),  "dim_payment")
write(build_code_dim(spark, dims["vendor"],   "vendor_id",   "vendor_name"),   "dim_vendor")
write(build_code_dim(spark, dims["ratecode"], "ratecode_id", "ratecode_desc"), "dim_ratecode")

In [0]:
write(build_dim_location(zones), "dim_location")
write(build_dim_time(spark), "dim_time")

start, end = date_bounds(trips)
print("dim_date range:", start, "->", end)
write(build_dim_date(spark, start, end), "dim_date")

In [0]:
write(build_fact_trip(trips), "fact_trip")

In [0]:
# role-playing views for Power BI (pickup / dropoff location)
create_role_playing_views(spark, catalog)
spark.sql(f"SHOW VIEWS IN {GOLD}").show()

In [0]:
# check
fact = spark.table(f"{GOLD}.fact_trip")

n_fact, n_silver = fact.count(), trips.count()
print(f"fact_trip {n_fact:,} | silver.trips {n_silver:,}")
assert n_fact == n_silver, "fact_trip row count differs from silver.trips"